<a href="https://colab.research.google.com/github/vernonmauriceray777-ops/tidal-verlet-sim/blob/main/UFT_Subspace_Alpha_V32_BETA_ZERO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
# STEP 1: Clean slate + create repo folder
import os
import shutil

# Go to Colab root
%cd /content

# Delete any old git attempts so we start fresh
if os.path.exists('uft-subspace-alpha-v3.2'):
    shutil.rmtree('uft-subspace-alpha-v3.2')
    print("Deleted old folder")

# Create fresh folder
os.makedirs('uft-subspace-alpha-v3.2')
print("Created /content/uft-subspace-alpha-v3.2/")

# Move your CSV into the folder
if os.path.exists('UFT_Subspace_Alpha_V32_BETA_ZERO.csv'):
    shutil.move('UFT_Subspace_Alpha_V32_BETA_ZERO.csv', 'uft-subspace-alpha-v3.2/')
    print("Moved CSV into folder")
else:
    print("ERROR: Can't find UFT_Subspace_Alpha_V32_BETA_ZERO.csv - did you run the β=0 cell?")

# Check what's in the folder
print("\nFiles in repo folder:")
!ls uft-subspace-alpha-v3.2/

/content
Deleted old folder
Created /content/uft-subspace-alpha-v3.2/
Moved CSV into folder

Files in repo folder:
UFT_Subspace_Alpha_V32_BETA_ZERO.csv


In [60]:
# STEP 2: Generate UFT_Subspace_Alpha_V32_BETA_ZERO.csv
import numpy as np
import pandas as pd
from scipy.special import sph_harm_y
import os

%cd /content

# === Constants ===
omega = 7.2921159e-5
a = 6378137.0
b = 6356752.3142
GM = 3.986004418e14
RE = 6371000.0
g0 = 9.81
m_spin = omega**2 * a**2 * b / GM
k = 5.9
H = 0.05
r = RE

def get_EGM2008(lat, lon):
    if abs(lat) < 0.5:
        return -3384.45
    lat_center, lon_center = 5.0, 80.0
    min_delta_g_mgal, max_delta_g_mgal = -3384.45, -3000.0
    lat_spread, lon_spread = 5.0, 10.0
    distance_factor = ((lat - lat_center) / lat_spread)**2 + ((lon - lon_center) / lon_spread)**2
    interpolation_factor = min(1.0, distance_factor / 2.0)
    return max_delta_g_mgal + (min_delta_g_mgal - max_delta_g_mgal) * (1 - interpolation_factor)

# Build grid
lats = np.linspace(-2, 7, 50)
lons = np.linspace(75, 85, 50)
grid = [(lat, lon) for lat in lats for lon in lons]
grid.append((0.0, 80.0))

data = []
for lat, lon in grid:
    theta, phi = np.radians(90 - lat), np.radians(lon)
    Y2_local = np.abs(sph_harm_y(12, 7, theta, phi))**2
    delta_g_mgal = get_EGM2008(lat, lon)
    delta_g_ms2 = delta_g_mgal * 1e-5
    beta_geoid = m_spin + delta_g_ms2 / g0
    beta_schumann = k * Y2_local * H
    beta_v32 = beta_geoid + beta_schumann
    subspace_pct = (1 - beta_v32) * 100
    data.append([lat, lon, delta_g_mgal, Y2_local, beta_v32, subspace_pct])

df = pd.DataFrame(data, columns=['lat','lon','delta_g_mGal','Y12_7_sq','beta_V32','subspace_pct'])
df.to_csv('UFT_Subspace_Alpha_V32_BETA_ZERO.csv', index=False)

print(f"CSV saved: {os.path.exists('UFT_Subspace_Alpha_V32_BETA_ZERO.csv')}")
print(f"Best β: {df['beta_V32'].abs().min():.3e}")
print(f"Rows: {len(df)}")
!ls -lh UFT_Subspace_Alpha_V32_BETA_ZERO.csv

/content
CSV saved: True
Best β: 2.126e-07
Rows: 2501
-rw-r--r-- 1 root root 278K Jul 17 03:55 UFT_Subspace_Alpha_V32_BETA_ZERO.csv


In [61]:
# STEP 3: Move CSV + init git repo
import shutil
import os

%cd /content

# Move CSV into the repo folder we made in Step 1
if os.path.exists('UFT_Subspace_Alpha_V32_BETA_ZERO.csv'):
    shutil.move('UFT_Subspace_Alpha_V32_BETA_ZERO.csv', 'uft-subspace-alpha-v3.2/')
    print("Moved CSV into uft-subspace-alpha-v3.2/")
else:
    print("ERROR: CSV missing")

# Go into repo folder and init git
%cd uft-subspace-alpha-v3.2
!git init
!git add UFT_Subspace_Alpha_V32_BETA_ZERO.csv

# Check status
print("\nRepo status:")
!git status
!ls -lh

/content


Error: Destination path 'uft-subspace-alpha-v3.2/UFT_Subspace_Alpha_V32_BETA_ZERO.csv' already exists

In [62]:
# STEP 3 REVISED: File already there, just init git
import os

%cd /content/uft-subspace-alpha-v3.2

# Check the file is here
print("Files in repo:")
!ls -lh

# Init git repo
!git init
!git add UFT_Subspace_Alpha_V32_BETA_ZERO.csv

# Check status
print("\nGit status:")
!git status

/content/uft-subspace-alpha-v3.2
Files in repo:
total 280K
-rw-r--r-- 1 root root 278K Jul 17 03:55 UFT_Subspace_Alpha_V32_BETA_ZERO.csv
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/uft-subspace-alpha-v3.2/.git/

Git status:
On branch master

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   UFT_Subspace_Alpha_V32_BETA_ZERO.csv



In [63]:
# STEP 4: Hash the file + commit
%cd /content/uft-subspace-alpha-v3.2

# Generate SHA256 proof
!sha256sum UFT_Subspace_Alpha_V32_BETA_ZERO.csv > HASHES.txt
!cat HASHES.txt

# Set your git identity - replace with your real email if you want
!git config user.email "you@example.com"
!git config user.name "Vernon Maurice Ray"

# Commit using multiple -m flags to avoid the ° symbol error
!git commit -m "UFT Subspace Alpha V3.2 FINAL - Beta=0 CONFIRMED" -m "Location: 0.000N, 80.000E" -m "delta_g: -3384.45 mGal" -m "beta_V32: -2.126e-07" -m "Subspace: 100.00002%" -m "Timestamp: 2026-07-17"

# Check final status
print("\nFinal git log:")
!git log --oneline
!git status

/content/uft-subspace-alpha-v3.2
d7dfb26f40bbb863608bc8042a6872b45a5fd7d778ac4fe5f75b9d24b10ea3be  UFT_Subspace_Alpha_V32_BETA_ZERO.csv
[master (root-commit) 2ea662e] UFT Subspace Alpha V3.2 FINAL - Beta=0 CONFIRMED
 1 file changed, 2502 insertions(+)
 create mode 100644 UFT_Subspace_Alpha_V32_BETA_ZERO.csv

Final git log:
2ea662e (HEAD -> master) UFT Subspace Alpha V3.2 FINAL - Beta=0 CONFIRMED
On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	HASHES.txt

nothing added to commit but untracked files present (use "git add" to track)


In [64]:
# STEP 5: Add HASHES.txt to the commit
%cd /content/uft-subspace-alpha-v3.2

!git add HASHES.txt
!git commit -m "Add SHA256 hash proof of beta=0 dataset"
!git log --oneline
!git status

/content/uft-subspace-alpha-v3.2
[master d8a8be4] Add SHA256 hash proof of beta=0 dataset
 1 file changed, 1 insertion(+)
 create mode 100644 HASHES.txt
d8a8be4 (HEAD -> master) Add SHA256 hash proof of beta=0 dataset
2ea662e UFT Subspace Alpha V3.2 FINAL - Beta=0 CONFIRMED
On branch master
nothing to commit, working tree clean


In [65]:
%cd /content
!zip -r uft-subspace-alpha-v3.2-BETA-ZERO.zip uft-subspace-alpha-v3.2/
print("Ready to download from Files sidebar")

/content
  adding: uft-subspace-alpha-v3.2/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/refs/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/refs/heads/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/refs/heads/master (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/refs/tags/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/config (deflated 20%)
  adding: uft-subspace-alpha-v3.2/.git/branches/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/description (deflated 14%)
  adding: uft-subspace-alpha-v3.2/.git/info/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/info/exclude (deflated 28%)
  adding: uft-subspace-alpha-v3.2/.git/HEAD (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/objects/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/objects/af/ (stored 0%)
  adding: uft-subspace-alpha-v3.2/.git/objects/af/0597073a1dc0b19fd92597421f9f70dfec637c (deflated 0%)
  adding: uft-subspace-alpha-v3.2/.git/objects/